# PolicyLab: Can logged clicks justify a recommendation-policy change?
**Ruturajsinh Zala · Root2Raj** | AI-assisted personal Data Science + Data Architecture research

4,530,676 real ZOZOTOWN impressions, two historical experiment policies, 34 Men-campaign actions, seven days (24–30 November 2019). This reproducible evidence audit deliberately concludes **HOLD**. It exposes a reference-policy mismatch and an inconclusive final improvement, rather than manufacturing a positive result.

[Interactive lab](https://root2raj.ruturaj1zala123.chatgpt.site/projects/policylab/) · [GitHub](https://github.com/Root2Raj-Zala/root2raj-policylab) · [Official data](https://research.zozo.com/data.html) · [Paper](https://arxiv.org/abs/2008.07146)

Data/derived audit: CC BY 4.0, Saito, Aihara, Matsutani & Narita. Original code MIT; upstream production priors Apache 2.0. Changes: position filtering, fitting, weighting and aggregation. No endorsement, client outcome or operational deployment is claimed.

The default run audits the **frozen real final-evaluation rows**, executes the estimator and recovery tests, verifies every packaged evidence hash and reruns the synthetic oracle study. Full 1.485GB raw-source processing is optional in the final section. These scopes are kept distinct.

In [ ]:
from pathlib import Path
import os,sys,json,hashlib,base64,io,zipfile,subprocess,importlib.util
ROOT=Path('/kaggle/working/policylab-audit') if Path('/kaggle').exists() else Path.cwd()/'.cache/policylab-notebook-audit'
ROOT.mkdir(parents=True,exist_ok=True)
for key,sub in [('TEMP','tmp'),('TMP','tmp'),('PYTHONPYCACHEPREFIX','pycache'),('MPLCONFIGDIR','matplotlib'),('PIP_CACHE_DIR','pip')]:
    p=ROOT/'.cache'/sub;p.mkdir(parents=True,exist_ok=True);os.environ[key]=str(p)
os.environ['OMP_NUM_THREADS']='2';os.environ['OPENBLAS_NUM_THREADS']='2'
print('CPU-only, no paid API. Working directory:',ROOT)


## 1. Download the immutable, hashed evidence package
Kaggle limits notebook source to 1MB. This version downloads the same offline package from an exact GitHub commit and verifies its SHA-256 before extraction. Internet must be enabled; CPU and free public endpoints are sufficient. The complete offline self-contained notebook is also on GitHub.

In [ ]:
PACKAGE_SHA256='56d684f36b692361718ef576451a494866cd11cc43294629e733a41828531921'
import urllib.request
PACKAGE_URL='https://raw.githubusercontent.com/Root2Raj-Zala/root2raj-policylab/232f2e1d2d1db8f1ea7175077bc0f1363378cc0a/notebooks/source-audit.zip'
payload=urllib.request.urlopen(PACKAGE_URL,timeout=120).read()
assert hashlib.sha256(payload).hexdigest()==PACKAGE_SHA256
with zipfile.ZipFile(io.BytesIO(payload)) as z:
    assert all(not Path(p).is_absolute() and '..' not in Path(p).parts for p in z.namelist())
    z.extractall(ROOT)
sys.path.insert(0,str(ROOT/'src'))
if importlib.util.find_spec('duckdb') is None:
    subprocess.check_call([sys.executable,'-m','pip','install','duckdb==1.5.6','--target',str(ROOT/'.deps'),'--disable-pip-version-check'])
    sys.path.insert(0,str(ROOT/'.deps'))
print('Verified archive:',PACKAGE_SHA256)


## 2. Execute meaningful scientific and recovery tests
Tests use separate fixtures with known answers. They cover DR misspecification identities, normalization and positivity failures, independent paired uncertainty, future outcomes, official-loop Monte Carlo parity, legitimate duplicate preservation and an injected pre-commit interruption. Fixtures never inflate the real-data count.

In [ ]:
import unittest
suite=unittest.defaultTestLoader.discover(str(ROOT/'tests'))
result=unittest.TextTestRunner(verbosity=1).run(suite)
assert result.wasSuccessful()
print('Tests passed:',result.testsRun)


## 3. Verify frozen real-data evidence independently
Recompute IPS, SNIPS, DR, paired differences and ESS from the actual 41,536 held-out logged rows. Missing raw files are explicitly reported as not rechecked; their published hashes remain in the manifest.

In [ ]:
import numpy as np,pandas as pd
from policylab.verify import verify
checks=verify(ROOT)
assert all(v is None or v for v in checks.values())
display(pd.Series(checks,name='passed (None = raw files not downloaded)'))
integrity=json.loads((ROOT/'artifacts/integrity.json').read_text())
for name,h in integrity['files'].items():
    assert hashlib.sha256((ROOT/'artifacts'/name).read_bytes()).hexdigest()==h,name
print('Evidence hashes verified:',len(integrity['files']))
audit=pd.read_csv(ROOT/'artifacts/test-audit.csv.gz')
assert not any(c.startswith('user_feature') for c in audit)
display(audit.drop(columns='source_sha').head(5))


## 4. Selection discipline and final evidence
First four UTC days fit the fixed-strength empirical-Bayes reward model and candidate policies. Day five chooses one of four candidates by its paired DR lower bound. The final two days do not select the winner. The selected policy is an action-only rule with 50% uniform exploration. Contextual candidates were considered in validation; their final outcomes are not used for selection.

In [ ]:
metrics=json.loads((ROOT/'artifacts/metrics.json').read_text())
selection=json.loads((ROOT/'artifacts/selection.json').read_text())
print(json.dumps({k:selection[k] for k in ['selected','train_days','validation_days','test_days']},indent=2))
display(pd.DataFrame(metrics['test']['estimates']).T)
print('Paired DR vs uniform:',metrics['test']['paired_dr_vs_uniform'])
print('Decision:',metrics['review'])


In [ ]:
from IPython.display import SVG,display
display(SVG(filename=str(ROOT/'artifacts/benchmark.svg')))
display(SVG(filename=str(ROOT/'artifacts/clipping.svg')))


## 5. Challenge the reference
The documented production-prior BernoulliTS algorithm is reconstructed using one million Monte Carlo draws. A test compares our vectorization against the literal upstream loop. But algorithm parity is different from matching the historical logging policy: the reconstructed fixed-prior first-position distribution differs from observed exposure frequencies. Its estimate also misses the observed BTS CTR under the frozen diagnostic. The project therefore **does not certify the learned policy value**. No single cause is established; data/prior alignment, policy dynamics and population differences require further investigation.

In [ ]:
reference=json.loads((ROOT/'artifacts/reference.json').read_text())
review=json.loads((ROOT/'artifacts/reference-review.json').read_text())
print(json.dumps(review,indent=2))
display(pd.read_csv(ROOT/'artifacts/reference-action-diagnostics.csv').query('position == 1').sort_values('n',ascending=False).head(10))
print('Observed BTS CTR',reference['observed_bts'],'IPS reconstructed target',reference['estimates']['IPS']['value'])


## 6. Warehouse recovery evidence
The complete source replay and every committed Parquet hash were checked locally. The default notebook displays that receipt; the tests independently inject and recover a small interruption. A source SHA + original row offset identifies an ingestion record. Equal-valued rows are retained. This is a local single-writer visibility protocol, not a global streaming exactly-once claim.

In [ ]:
print(json.dumps(json.loads((ROOT/'artifacts/replay.json').read_text()),indent=2))
display(pd.read_csv(ROOT/'artifacts/reconciliation.csv'))
print('Protocol SHA:',metrics['protocol_sha256'])


## 7. Separate synthetic oracle experiment
Here the true unobserved action rewards are known by construction. A deliberately biased reward model illustrates why DM can be wrong and how weak overlap raises IPS/DR uncertainty. These synthetic records are never called ZOZO impressions. The simulation tests behavior under its assumptions, not real-data validity.

In [ ]:
from policylab.simulation import run as simulate
simout=ROOT/'simulation-rerun';simout.mkdir(exist_ok=True)
simulation=simulate(simout,repetitions=250,n=4000)
display(simulation)
expected=pd.read_csv(ROOT/'artifacts/simulation-summary.csv')
np.testing.assert_allclose(simulation.select_dtypes('number'),expected.select_dtypes('number'),rtol=1e-9,atol=1e-10)
print('500 synthetic repetitions reproduced; separate from real counts')


## 8. Optional full-source rerun
Leave the switch off for the evidence audit. Turn it on to download both complete Men logs from the pinned official release (about 1.485GB), ingest every row and rerun the protocol. Internet access and several GB of free disk are needed; CPU is sufficient. No paid API or GPU is used. This cell does not claim a raw-source rerun unless the switch is enabled.

In [ ]:
RUN_FULL_SOURCE=False
if RUN_FULL_SOURCE:
    subprocess.check_call([sys.executable,str(ROOT/'scripts/download.py')],cwd=ROOT)
    from policylab.pipeline import run as full_run
    rerun=full_run(ROOT)
    assert rerun['rows_all_positions']==4530676
    np.testing.assert_allclose(rerun['test']['estimates']['DR']['value'],metrics['test']['estimates']['DR']['value'],rtol=1e-9)
    print('FULL RAW-SOURCE RERUN PASSED')
else:
    print('Frozen real-audit scope completed. Full raw-source rerun was not requested.')


## What this result supports
Inspecting assumptions, estimation, failure cases and source lineage is useful portfolio evidence. Seven historical days, unavailable users, Monte Carlo propensities, an unresolved reference mismatch and conditional intervals cannot establish a production gain. Marginal item-position probabilities do not establish whole-slate effects. A large row count cannot repair those gaps. See the packaged protocol, architecture runbook, model card and plain-language walkthrough for the complete reasoning.

**Attribution:** Saito, Y., Aihara, S., Matsutani, M., & Narita, Y. (2021), *Open Bandit Dataset and Pipeline*. Data: CC BY 4.0; transformed audit/aggregates retain attribution; upstream priors/reference Apache 2.0. No endorsement. Original implementation developed with AI assistance for Ruturajsinh Zala (Root2Raj), MIT.